# 08 — Noise-floor control study

Re-embeds round and observed-geometry receptive fields in each cell's own
signal-free-lag noise, refits with the unchanged pipeline, and reports
per-cell bootstrap intervals, a population-level noise-floor test, a
cross-noise control, and fitted-model orientation estimates. Output written
to `outputs/noise_floor/`.

In [ ]:
from pathlib import Path
import sys, os, pickle, inspect, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from joblib import Parallel, delayed

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

from rf_analysis.sparse_noise import (fit_rf_by_order, gaussian_deriv_m1,
                                      batch_reconstruct_strf)

cache_dir    = REPO_ROOT / 'data' / 'cache'
lists_dir    = REPO_ROOT / 'data' / 'experiment_lists'
gallery_dir  = REPO_ROOT / 'derived_data' / 'm1_cells'
DATASET_PATH = gallery_dir / 'm1_neuron_dataset.pkl'
OUT_DIR      = REPO_ROOT / 'outputs' / 'noise_floor'
STRF_CACHE   = OUT_DIR / 'strf_cache'
OUT_DIR.mkdir(parents=True, exist_ok=True); STRF_CACHE.mkdir(exist_ok=True)

# Fitting protocol: read from the library, not restated
SMOOTH_SIGMA    = inspect.signature(fit_rf_by_order).parameters['smooth_sigma'].default
N_RANDOM_STARTS = inspect.signature(fit_rf_by_order).parameters['n_random_starts'].default

# Simulation protocol
N_NULL      = 100      # round (kappa=1) draws per cell
N_BOOT      = 100      # planted-observed draws per cell
N_ROT       = 50       # planted-observed geometry at a uniformly random orientation
LAG_GAP     = 4        # clean lags start >= best_lag + LAG_GAP presentations (>= 1 s)
LEAK_R_MAX  = 0.30     # |corr| with the smoothed best-lag map above this = leakage
USE_FLIPS   = True     # augment the noise pool with the 3 reflections of each map
N_POP       = 5000     # null populations for the population-level test
N_JOBS      = -1
SEED        = 20260926

with open(DATASET_PATH, 'rb') as f:
    dataset = pickle.load(f)
assert len(dataset) == 31, f'Expected 31 verified m=1 cells, got {len(dataset)}'
cells = pd.DataFrame([{k: r[k] for k in
    ['name', 'cell_id', 'container_id', 'pixel_size_deg', 'r_squared', 'kappa_dir',
     'sigma_deg', 'sigma_phi_deg', 'sigma_orth_deg', 'x0_px', 'y0_px',
     'amplitude', 'theta_envelope']} for r in dataset])
rf_raw = {int(r['cell_id']): np.asarray(r['rf_raw'], float) for r in dataset}
GRID = next(iter(rf_raw.values())).shape
print(f'{len(cells)} cells, {cells.container_id.nunique()} containers, grid {GRID}')
print(f'smooth_sigma={SMOOTH_SIGMA} px, starts=1+{N_RANDOM_STARTS}, '
      f'N_NULL={N_NULL}, N_BOOT={N_BOOT}')

## Session mapping

In [ ]:
exp_df = pd.read_csv(lists_dir / 'all_l23_excitatory_experiments.csv')
session_c = exp_df[exp_df['session_type'].str.contains('C')]
container_to_exp = dict(zip(session_c['experiment_container_id'], session_c['id']))

def get_session_c_dataset(container_id):
    # Only hit the Allen SDK on a cache miss (see Part 2); the 13 containers
    # behind the 31 verified cells all have an STRF cache committed here.
    from allensdk.core.brain_observatory_cache import BrainObservatoryCache
    boc = BrainObservatoryCache(manifest_file=str(cache_dir / 'manifest.json'))
    return boc.get_ophys_experiment_data(container_to_exp[int(container_id)])

missing = set(cells.container_id) - set(container_to_exp)
assert not missing, f'No Session C experiment for containers {missing}'
print(f'{cells.container_id.nunique()} containers mapped')

## Full STRFs (canonical lineage)

One ridge fit per container, cached to disk. The best-lag slice must
reproduce the stored `rf_raw` for every cell, or the lineage check below
fails and stops the notebook.

In [ ]:
strf, best_lag, lam = {}, {}, {}
t0 = time.time()
for cont, grp in cells.groupby('container_id'):
    cache = STRF_CACHE / f'strf_{cont}.npz'
    if cache.exists():
        z = np.load(cache)
        out = {int(c): {'strf': z[f's{c}'], 'best_lag': int(z[f'b{c}']),
                        'ridge_lambda': float(z[f'l{c}'])} for c in grp.cell_id}
        src = 'cache'
    else:
        out = batch_reconstruct_strf(get_session_c_dataset(cont), grp.cell_id.tolist())
        np.savez(cache, **{f's{c}': v['strf'] for c, v in out.items()},
                        **{f'b{c}': v['best_lag'] for c, v in out.items()},
                        **{f'l{c}': v['ridge_lambda'] for c, v in out.items()})
        src = 'ridge'
    for c, v in out.items():
        strf[c], best_lag[c], lam[c] = v['strf'], v['best_lag'], v['ridge_lambda']
    print(f'  container {cont}: {len(out)} cells ({src})  {time.time()-t0:.0f}s')

bad = []
for c in cells.cell_id:
    rec = strf[c][best_lag[c]]
    if rec.shape != rf_raw[c].shape or not np.allclose(rec, rf_raw[c], rtol=1e-4,
                                                       atol=1e-6 * np.abs(rf_raw[c]).max()):
        bad.append(c)
assert not bad, f'STRF best-lag map does not reproduce stored rf_raw for {bad}'
cells['best_lag'] = cells.cell_id.map(best_lag)
print(f'\nLINEAGE CHECK PASSED: 31/31 best-lag maps reproduce stored rf_raw.')
print('best-lag distribution:', cells.best_lag.value_counts().sort_index().to_dict())

## Signal-free lags and leakage check

Candidate noise lags are `lag >= best_lag + LAG_GAP`; any lag whose
correlation with the smoothed best-lag map exceeds `LEAK_R_MAX` is rejected
as carrying residual signal.

In [ ]:
def smooth(a):
    return gaussian_filter(a, SMOOTH_SIGMA)

rows, noise_maps = [], {}
for c in cells.cell_id:
    ref = smooth(strf[c][best_lag[c]]).ravel()
    keep = []
    for L in range(best_lag[c] + LAG_GAP, strf[c].shape[0]):
        r = float(np.corrcoef(ref, smooth(strf[c][L]).ravel())[0, 1])
        ok = abs(r) <= LEAK_R_MAX
        rows.append({'cell_id': c, 'lag': L, 'r_with_best': r, 'accepted': ok})
        if ok:
            keep.append(strf[c][L])
    noise_maps[c] = keep
lag_tab = pd.DataFrame(rows)

cells['n_noise_lags'] = cells.cell_id.map(lambda c: len(noise_maps[c]))
cells['noise_rms']    = cells.cell_id.map(
    lambda c: float(np.median([np.sqrt(np.mean(m**2)) for m in noise_maps[c]]))
    if noise_maps[c] else np.nan)
cells['signal_peak']  = cells.cell_id.map(lambda c: float(np.abs(rf_raw[c]).max()))

print(f'candidate lags: {len(lag_tab)}   accepted: {int(lag_tab.accepted.sum())}   '
      f'rejected for leakage: {int((~lag_tab.accepted).sum())}')
print(f'|r| of accepted lags with best-lag map: median '
      f'{lag_tab.loc[lag_tab.accepted, "r_with_best"].abs().median():.3f}')
no_noise = cells[cells.n_noise_lags == 0]
if len(no_noise):
    print('cells with no own clean lag (noise scale falls back to pooled median SNR):',
          no_noise.name.tolist())
print(cells[['name', 'best_lag', 'n_noise_lags', 'noise_rms', 'signal_peak']]
      .round(5).to_string(index=False))

In [ ]:
# Cells with no accepted lag of their own get a noise scale from the population's
# median peak-to-noise ratio, applied to their own peak.  Reported, not hidden.
med_snr = float(np.nanmedian(cells.signal_peak / cells.noise_rms))
fill = cells.noise_rms.isna()
cells.loc[fill, 'noise_rms'] = cells.loc[fill, 'signal_peak'] / med_snr
cells['noise_scale_source'] = np.where(fill, 'pooled SNR', 'own lags')
print(f'median peak/noise = {med_snr:.2f};  pooled fallback used for {int(fill.sum())} cells')

# Noise pool: every accepted map from every cell, unit RMS, plus reflections.
pool = []
for c in cells.cell_id:
    for m in noise_maps[c]:
        m = m / np.sqrt(np.mean(m**2))
        pool += [m, m[::-1, :], m[:, ::-1], m[::-1, ::-1]] if USE_FLIPS else [m]
pool = np.stack(pool)
print(f'noise pool: {len(pool)} unit-RMS maps  (grid {pool.shape[1:]})')

## Rebuild and verify the fitted model (geometry check)

The rebuilt m=1 model must correlate with the smoothed map at |r|² ≈ the
stored R²; a mismatch means the geometry was misread and the check below
stops the notebook.

In [ ]:
H, W = GRID
YY, XX = np.mgrid[0:H, 0:W]
XY = (XX.astype(float), YY.astype(float))

def u_axis_rad(row):
    th = row.theta_envelope if row.kappa_dir <= 1 else row.theta_envelope - 90.0
    return np.deg2rad(th)

def fitted_model(row):
    px = row.pixel_size_deg
    return gaussian_deriv_m1(XY, row.amplitude, row.x0_px, row.y0_px,
                             row.sigma_phi_deg / px, row.sigma_orth_deg / px,
                             u_axis_rad(row), 0.0).reshape(H, W)

chk = []
for _, row in cells.iterrows():
    r = np.corrcoef(fitted_model(row).ravel(), smooth(rf_raw[row.cell_id]).ravel())[0, 1]
    chk.append(r**2)
cells['r2_rebuilt'] = chk
cells['r2_gap'] = (cells.r2_rebuilt - cells.r_squared).abs()
worst = cells.r2_gap.max()
print(f'rebuilt-model R2 vs stored R2: max |gap| = {worst:.4f}')
assert worst < 0.01, cells.loc[cells.r2_gap >= 0.01, ['name', 'r_squared', 'r2_rebuilt']]
print('GEOMETRY CHECK PASSED')

## Templates and the simulation

Templates are planted in raw (pre-smoothing) space and scaled so their
post-smoothing energy matches each cell's fitted model, giving every
synthetic map the cell's own SNR. **Null:** round, at the cell's own size.
**Bootstrap:** the cell's observed σ_φ, σ_⊥. Noise is a random pool map
scaled to the cell's own noise RMS.

In [ ]:
S = SMOOTH_SIGMA

def raw_axis(sig_fit_px):
    assert sig_fit_px > S, 'fitted axis at or below the smoothing scale'
    return float(np.sqrt(sig_fit_px**2 - S**2))

def u_from_params(theta_deg, kappa_dir):
    # fit_rf_by_order stores theta after the major/minor swap; undo it
    return (theta_deg if kappa_dir <= 1 else theta_deg - 90.0) % 180.0

GEOM = {}
for _, row in cells.iterrows():
    px = row.pixel_size_deg
    GEOM[row.cell_id] = dict(
        x0=row.x0_px, y0=row.y0_px, u_deg=np.rad2deg(u_axis_rad(row)) % 180,
        su_obs=raw_axis(row.sigma_phi_deg / px), sv_obs=raw_axis(row.sigma_orth_deg / px),
        s_round=raw_axis(row.sigma_deg / px),
        E=float(np.sum(fitted_model(row)**2)), px=px, noise=row.noise_rms)

def template(c, kind, u_deg):
    g = GEOM[c]
    su, sv = (g['s_round'], g['s_round']) if kind == 'round' else (g['su_obs'], g['sv_obs'])
    T = gaussian_deriv_m1(XY, 1.0, g['x0'], g['y0'], su, sv,
                          np.deg2rad(u_deg), 0.0).reshape(H, W)
    return T * np.sqrt(g['E'] / float(np.sum(smooth(T)**2)))

def one_fit(c, kind, i):
    rng = np.random.default_rng([SEED, int(c) % 2**31, {'round': 0, 'boot': 1, 'rot': 2}[kind], i])
    g = GEOM[c]
    u_plant = float(rng.uniform(0, 180)) if kind == 'rot' else g['u_deg']
    m = template(c, kind, u_plant) + g['noise'] * pool[rng.integers(len(pool))]
    p, q = fit_rf_by_order(np.clip(m, 0, None), np.clip(-m, 0, None), m=1,
                           pixel_size_deg=g['px'], smooth_sigma=S,
                           n_random_starts=N_RANDOM_STARTS,
                           seed=int(rng.integers(2**31)))
    kd = p.get('kappa_dir', np.nan)
    u_rec = u_from_params(p['theta'], kd) if np.isfinite(kd) else np.nan
    return {'cell_id': c, 'kind': kind, 'draw': i, 'kappa_dir': kd,
            'r_squared': q['r_squared'], 'u_planted': u_plant, 'u_recovered': u_rec}

# timing probe before committing to the full run
t = time.time(); one_fit(int(cells.cell_id.iloc[0]), 'round', 0); dt = time.time() - t
n_fits = len(cells) * (N_NULL + N_BOOT + N_ROT)
print(f'one fit {dt:.2f}s -> {n_fits} fits ~ {n_fits*dt/60:.0f} CPU-min '
      f'(divide by cores in use)')

In [ ]:
SIM_CSV = OUT_DIR / f'sims_null{N_NULL}_boot{N_BOOT}_rot{N_ROT}.csv'
if SIM_CSV.exists():
    sims = pd.read_csv(SIM_CSV)
    print(f'loaded {len(sims)} fits from {SIM_CSV.name}')
else:
    jobs = [(c, k, i) for c in cells.cell_id
            for k, n in (('round', N_NULL), ('boot', N_BOOT), ('rot', N_ROT))
            for i in range(n)]
    t = time.time()
    sims = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(one_fit)(*j) for j in jobs))
    sims.to_csv(SIM_CSV, index=False)
    print(f'{len(sims)} fits in {(time.time()-t)/60:.1f} min -> {SIM_CSV.name}')

## Results

Calibration first: bootstrap-fit R² should land close to each cell's stored
R² if the injected noise level is right.

In [ ]:
boot = sims[sims.kind == 'boot']; null = sims[sims.kind == 'round']
cal = boot.groupby('cell_id').r_squared.median().rename('r2_boot_median')
cells = cells.drop(columns=[c for c in ['r2_boot_median'] if c in cells]).merge(
    cal, left_on='cell_id', right_index=True)
cal_ratio = (cells.r2_boot_median / cells.r_squared)
print(f'CALIBRATION  median(bootstrap R2) / stored R2: median {cal_ratio.median():.2f} '
      f'(IQR {cal_ratio.quantile(.25):.2f}-{cal_ratio.quantile(.75):.2f})')
print('  ~1 = synthetic maps are as noisy as the real ones; <1 = noisier (conservative);'
      ' >1 = cleaner (null too narrow).')

In [ ]:
def q(x, a): return float(np.nanpercentile(x, a))

per = []
for _, row in cells.iterrows():
    c, k_obs = row.cell_id, row.kappa_dir
    kn = null.loc[null.cell_id == c, 'kappa_dir'].dropna().values
    kb = boot.loc[boot.cell_id == c, 'kappa_dir'].dropna().values
    ln, lo = np.abs(np.log(kn)), abs(np.log(k_obs))
    per.append({
        'name': row['name'], 'cell_id': c, 'kappa_dir': k_obs, 'r_squared': row.r_squared,
        'null_lo': q(kn, 2.5), 'null_hi': q(kn, 97.5), 'null_median': q(kn, 50),
        'p_vs_round': (1 + np.sum(ln >= lo)) / (1 + len(ln)),
        'ci_lo': q(kb, 2.5), 'ci_hi': q(kb, 97.5), 'boot_median': q(kb, 50),
        'recovery_ratio': q(kb, 50) / k_obs,
    })
per = pd.DataFrame(per).sort_values('kappa_dir').reset_index(drop=True)

# Benjamini-Hochberg across the 31 cells
p = per.p_vs_round.values; o = np.argsort(p); m = len(p)
adj = np.minimum.accumulate((p[o] * m / np.arange(1, m + 1))[::-1])[::-1]
per.loc[o, 'p_bh'] = np.minimum(adj, 1)
per['distinct_from_round'] = per.p_bh < 0.05
print(per.round(3).to_string(index=False))

In [ ]:
# Population-level null: 31 round cells, each drawn from its own null
rng = np.random.default_rng(SEED)
null_by_cell = [null.loc[null.cell_id == c, 'kappa_dir'].dropna().values for c in per.cell_id]
def pop_stats(k):
    lk = np.log(k)
    return k.max() / k.min(), float(np.exp(lk.std(ddof=1))), \
           np.percentile(k, 90) / np.percentile(k, 10)
obs = pop_stats(per.kappa_dir.values)
sim = np.array([pop_stats(np.array([rng.choice(v) for v in null_by_cell]))
                for _ in range(N_POP)])
names = ['range factor', 'geometric SD', 'p90/p10']
print('=' * 74)
print('POPULATION NULL — 31 round cells at their own size, position and SNR')
print('=' * 74)
for j, nm in enumerate(names):
    pv = (1 + np.sum(sim[:, j] >= obs[j])) / (1 + N_POP)
    print(f'  {nm:14s} observed {obs[j]:5.2f}   null median {np.median(sim[:, j]):5.2f} '
          f'[95%: {np.percentile(sim[:, j], 2.5):.2f}-{np.percentile(sim[:, j], 97.5):.2f}]'
          f'   p = {pv:.4f}')
allnull = np.concatenate(null_by_cell)
print(f'\n  pooled null kappa_dir: median {np.median(allnull):.2f}, '
      f'95% {np.percentile(allnull, 2.5):.2f}-{np.percentile(allnull, 97.5):.2f}')
print(f'  cells distinguishable from round (BH q<0.05): '
      f'{int(per.distinct_from_round.sum())}/31')
print(f'  recovery ratio (bootstrap median / observed): median '
      f'{per.recovery_ratio.median():.2f}, IQR {per.recovery_ratio.quantile(.25):.2f}-'
      f'{per.recovery_ratio.quantile(.75):.2f}')
for nm in ['Winifred', 'Ottoline', 'Xiomara']:
    r = per[per.name == nm]
    if len(r):
        r = r.iloc[0]
        print(f'  {nm:9s} kappa_dir {r.kappa_dir:.2f}  95% CI {r.ci_lo:.2f}-{r.ci_hi:.2f}'
              f'   null {r.null_lo:.2f}-{r.null_hi:.2f}   q = {r.p_bh:.3f}')
print('=' * 74)

## Figure (supplementary)

In [ ]:
try:
    import figstyle as fs; fs.apply()
    C_OBS, C_NULL, C_INK, W_FIG = fs.RUST, fs.GREY_PALE, fs.INK, fs.W_FULL
except ImportError:
    fs = None; C_OBS, C_NULL, C_INK, W_FIG = '#C8552B', '#D8D5D4', '#3A3838', 7.2

from matplotlib.ticker import LogLocator, NullFormatter, FixedLocator, FixedFormatter

fig, ax = plt.subplots(1, 3, figsize=(W_FIG, W_FIG * 0.40),
                       gridspec_kw={'width_ratios': [1.7, 1.1, 1.1], 'wspace': 0.55})

# --- A: per-cell null band and observed CIs ---
a = ax[0]
x = np.arange(len(per))
a.vlines(x, per.null_lo, per.null_hi, color=C_NULL, lw=5, label='null 95% (round)')
a.vlines(x, per.ci_lo, per.ci_hi, color=C_OBS, lw=1.1, alpha=0.7)
a.scatter(x, per.kappa_dir, s=14, color=C_OBS, zorder=3, label='observed, 95% CI')
a.axhline(1, color=C_INK, lw=0.6, ls='--')
a.set_yscale('log'); a.set_xticks([])
a.set_xlabel(r'31 cells, ordered by $\kappa_{\mathrm{dir}}$')
a.set_ylabel(r'$\kappa_{\mathrm{dir}}$')
a.yaxis.set_major_locator(FixedLocator([0.1, 0.3, 1.0, 3.0]))
a.yaxis.set_major_formatter(FixedFormatter(['0.1', '0.3', '1', '3']))
a.yaxis.set_minor_formatter(NullFormatter())
a.legend(frameon=False, fontsize=7, loc='upper left')

# --- B: population range factor vs null ---
a = ax[1]
a.hist(sim[:, 0], bins=40, color=C_NULL)
a.axvline(obs[0], color=C_OBS, lw=1.8, label='observed')
a.set_xlabel('range factor across 31 cells')
a.set_ylabel('null populations')
a.set_xlim(0, np.percentile(sim[:, 0], 99.5))
a.legend(frameon=False, fontsize=7, loc='upper right')

# --- C: recovery of planted elongation ---
a = ax[2]
a.scatter(per.kappa_dir, per.boot_median, s=13, color=C_OBS)
lim = [0.9 * min(per.kappa_dir.min(), per.boot_median.min()),
       1.1 * max(per.kappa_dir.max(), per.boot_median.max())]
a.plot(lim, lim, color=C_INK, lw=0.6, ls='--')
a.set_xscale('log'); a.set_yscale('log'); a.set_xlim(lim); a.set_ylim(lim)
a.set_aspect('equal')
ticks = [0.5, 1.0, 2.0]
for axis in (a.xaxis, a.yaxis):
    axis.set_major_locator(FixedLocator(ticks))
    axis.set_major_formatter(FixedFormatter(['0.5', '1', '2']))
    axis.set_minor_locator(LogLocator(subs='all'))
    axis.set_minor_formatter(NullFormatter())
a.set_xlabel(r'planted $\kappa_{\mathrm{dir}}$')
a.set_ylabel(r'recovered (median)')

for a_, l in zip(ax, 'ABC'):
    (fs.panel_label(a_, l, dx=-0.22) if fs else a_.set_title(l, loc='left', fontweight='bold'))

fig.subplots_adjust(bottom=0.22)
if fs: fs.save(fig, 'figS_noise_floor', outdir=OUT_DIR)
else:  fig.savefig(OUT_DIR / 'figS_noise_floor.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Cross-noise control: plant each cell's geometry with noise drawn only from
# OTHER cells (never its own), scaled to its own noise level. Rules out a
# "shape attractor" from faint shared structure between template and noise.
from joblib import Parallel, delayed

# Per-cell pool of noise maps from OTHER cells only (unit RMS, with reflections)
other_pool = {}
for c in cells.cell_id:
    maps = []
    for d in cells.cell_id:
        if d == c:
            continue
        for m in noise_maps[d]:
            m = m / np.sqrt(np.mean(m ** 2))
            maps += [m, m[::-1, :], m[:, ::-1], m[::-1, ::-1]] if USE_FLIPS else [m]
    other_pool[c] = np.stack(maps)
print("cross-noise pools built; sizes:",
      {int(c): len(other_pool[c]) for c in list(cells.cell_id)[:3]}, "...")

def one_fit_cross(c, kind, i):
    # kind 'xround' = round template; 'xboot' = observed geometry. Noise from OTHER cells.
    rng = np.random.default_rng([SEED, int(c) % 2**31, {'xround': 3, 'xboot': 4}[kind], i])
    g = GEOM[c]
    T = template(c, 'round' if kind == 'xround' else 'boot', g['u_deg'])
    npool = other_pool[c]
    m = T + g['noise'] * npool[rng.integers(len(npool))]
    p, q = fit_rf_by_order(np.clip(m, 0, None), np.clip(-m, 0, None), m=1,
                           pixel_size_deg=g['px'], smooth_sigma=S,
                           n_random_starts=N_RANDOM_STARTS,
                           seed=int(rng.integers(2**31)))
    return {'cell_id': c, 'kind': kind, 'draw': i,
            'kappa_dir': p.get('kappa_dir', np.nan), 'r_squared': q['r_squared']}

XSIM = OUT_DIR / f'crossnoise_null{N_NULL}_boot{N_BOOT}.csv'
if XSIM.exists():
    xsims = pd.read_csv(XSIM)
    print(f'loaded {len(xsims)} cross-noise fits from {XSIM.name}')
else:
    xjobs = [(c, k, i) for c in cells.cell_id
             for k, n in (('xround', N_NULL), ('xboot', N_BOOT)) for i in range(n)]
    xsims = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(one_fit_cross)(*j) for j in xjobs))
    xsims.to_csv(XSIM, index=False)
    print(f'{len(xsims)} cross-noise fits -> {XSIM.name}')

xround = xsims[xsims.kind == 'xround']
xboot  = xsims[xsims.kind == 'xboot']

# --- 1. Population null: does the elongation spread exceed the cross-noise floor? ---
def iqr_ratio(k):
    k = k[np.isfinite(k)]; return np.percentile(k, 75) / np.percentile(k, 25)
obs_iqr = iqr_ratio(per.kappa_dir.values)
rng = np.random.default_rng(SEED + 7)
xnull_by_cell = [xround.loc[xround.cell_id == c, 'kappa_dir'].dropna().values for c in per.cell_id]
own_by_cell   = [null.loc[null.cell_id == c, 'kappa_dir'].dropna().values   for c in per.cell_id]
xpop = np.array([iqr_ratio(np.array([rng.choice(v) for v in xnull_by_cell])) for _ in range(5000)])
opop = np.array([iqr_ratio(np.array([rng.choice(v) for v in own_by_cell]))   for _ in range(5000)])
print('\n' + '=' * 70)
print('1. POPULATION NULL  (IQR ratio of kappa_dir across 31 cells)')
print(f'   observed                 {obs_iqr:.3f}')
print(f'   own-noise null   median  {np.median(opop):.3f}  95% [{np.percentile(opop,2.5):.3f}, {np.percentile(opop,97.5):.3f}]  p={(1+np.sum(opop>=obs_iqr))/5001:.4f}')
print(f'   cross-noise null median  {np.median(xpop):.3f}  95% [{np.percentile(xpop,2.5):.3f}, {np.percentile(xpop,97.5):.3f}]  p={(1+np.sum(xpop>=obs_iqr))/5001:.4f}')

# --- 2. Per-cell interval width: are own-noise intervals artificially tight? ---
def width68(df):
    return df.groupby('cell_id').kappa_dir.agg(
        lambda x: np.nanpercentile(x, 84) / np.nanpercentile(x, 16))
w_own, w_cross = width68(boot), width68(xboot)
w = pd.concat([w_own.rename('own'), w_cross.rename('cross')], axis=1).dropna()
print('\n2. PER-CELL 68% INTERVAL WIDTH  (ratio hi/lo; larger = wider/less certain)')
print(f'   own-noise   median {w.own.median():.3f}')
print(f'   cross-noise median {w.cross.median():.3f}')
print(f'   median (cross / own) per cell: {(w.cross / w.own).median():.3f}')
print('   >1 means cross-noise is WIDER, i.e. own-noise was optimistic (attractor present)')
print('   ~1 means no attractor; the two designs agree')

# --- 3. Bias: does either design recover planted kappa without bias? ---
def rec_ratio(df):
    m = df.groupby('cell_id').kappa_dir.median()
    return (m / per.set_index('cell_id').kappa_dir).median()
print('\n3. RECOVERY (median recovered / planted kappa_dir)')
print(f'   own-noise   {rec_ratio(boot):.3f}')
print(f'   cross-noise {rec_ratio(xboot):.3f}')
print('=' * 70)

## Orientation

The fitted m=1 model's differentiation axis u is resolved OFF→ON to give
φ_fit on [0°, 360°). Image-coordinate angles (clockwise on screen) are
converted to the displayed convention via φ_vis = −φ_fit mod 360°. The
random-orientation arm tests the fitter for a pull toward horizontal;
bootstrap draws give each cell's φ interval.

In [ ]:
def axial_diff(a, b):
    return np.abs(((np.asarray(a) - np.asarray(b)) + 90.0) % 180.0 - 90.0)

def to_vis(a_img, period=360.0):
    return (-np.asarray(a_img)) % period

def rayleigh(a_deg, axial=False):
    a = np.deg2rad(np.asarray(a_deg, float) * (2 if axial else 1)); n = len(a)
    if n == 0:
        return np.nan, np.nan
    R = float(np.abs(np.mean(np.exp(1j * a))))
    p = float(np.exp(np.sqrt(1 + 4*n + 4*(n**2 - (n*R)**2)) - (1 + 2*n)))
    return R, min(p, 1.0)

ori = []
for _, row in cells.iterrows():
    c = row.cell_id
    u_img = GEOM[c]['u_deg']
    unit = gaussian_deriv_m1(XY, 1.0, row.x0_px, row.y0_px, row.sigma_phi_deg / row.pixel_size_deg,
                             row.sigma_orth_deg / row.pixel_size_deg, np.deg2rad(u_img), 0.0)
    sgn = np.sign(np.corrcoef(unit, smooth(rf_raw[c]).ravel())[0, 1])
    phi_fit_img = (u_img + (0.0 if sgn > 0 else 180.0)) % 360.0
    b = boot[boot.cell_id == c]
    err = axial_diff(b.u_recovered, b.u_planted)
    rec = dataset[[r['cell_id'] for r in dataset].index(c)]
    phi_old = rec.get('phi_deg', np.nan)
    ori.append({'name': row['name'], 'cell_id': c, 'kappa_dir': row.kappa_dir,
                'phi_fit_img': phi_fit_img, 'phi_fit_vis': float(to_vis(phi_fit_img)),
                'axis_vis': float(to_vis(u_img, 180.0)),
                'phi_ci95_halfwidth': float(np.nanpercentile(err, 95)),
                'phi_boot_bias': float(np.nanmedian(((b.u_recovered - b.u_planted) + 90) % 180 - 90)),
                'phi_moment15_img': phi_old,
                'moment15_vs_fit': float(axial_diff(phi_old, u_img)) if pd.notna(phi_old) else np.nan,
                'theta_env_vs_fit': float(axial_diff(row.theta_envelope, u_img))})
ori = pd.DataFrame(ori)

print('TIGHT ORIENTATION ESTIMATES (displayed convention, CCW from rightward)')
print(ori[['name', 'kappa_dir', 'phi_fit_vis', 'phi_ci95_halfwidth', 'phi_boot_bias',
           'moment15_vs_fit', 'theta_env_vs_fit']].round(1).to_string(index=False))
hw = ori.phi_ci95_halfwidth
print(f'\n95% half-width: median {hw.median():.1f} deg, max {hw.max():.1f} deg '
      f'({ori.name[hw.idxmax()]})')
print(f'stored 15%-mask phi disagrees with fit axis by >45 deg in '
      f'{int((ori.moment15_vs_fit > 45).sum())}/31 cells '
      f'(median {ori.moment15_vs_fit.median():.1f} deg)')
for nm, tony in [('Valentina', '~0'), ('Bobby', '~60+'), ('Georgette', '~90+')]:
    r = ori[ori.name == nm]
    if len(r):
        r = r.iloc[0]
        print(f'  {nm:9s} axis {r.axis_vis:5.1f} deg  (phi_fit {r.phi_fit_vis:5.1f}, '
              f'+/-{r.phi_ci95_halfwidth:.1f})   Tony read {tony}')

In [ ]:
# Grid-bias test on the random-orientation arm
rot = sims[sims.kind == 'rot'].dropna(subset=['u_recovered'])
hz = lambda a: np.minimum(np.asarray(a) % 180, 180 - np.asarray(a) % 180)   # distance to horizontal
pull = hz(rot.u_planted) - hz(rot.u_recovered)          # > 0 = moved toward horizontal
err_rot = axial_diff(rot.u_recovered, rot.u_planted)
Rr, pr = rayleigh(rot.u_recovered, axial=True)
rng = np.random.default_rng(SEED + 1)
pull_boot = [np.mean(rng.choice(pull, len(pull))) for _ in range(2000)]

print('=' * 74)
print('ORIENTATION — fitter behaviour on uniformly planted angles')
print('=' * 74)
print(f'  {len(rot)} fits: median |error| {np.median(err_rot):.1f} deg, 95th pct {np.percentile(err_rot, 95):.1f} deg')
print(f'  pull toward horizontal: mean {np.mean(pull):+.2f} deg '
      f'[95% {np.percentile(pull_boot, 2.5):+.2f}, {np.percentile(pull_boot, 97.5):+.2f}]')
print(f'  recovered angles, axial Rayleigh: R = {Rr:.3f}, p = {pr:.3f}  (planted uniform)')

Rd, pd_ = rayleigh(ori.phi_fit_img)
Ra, pa = rayleigh(ori.phi_fit_img % 180, axial=True)
Rm, pm = rayleigh(ori.phi_moment15_img.dropna())
Re, pe = rayleigh(cells.theta_envelope, axial=True)
n_hz = int((hz(ori.phi_fit_img) < 30).sum())
print('\nOBSERVED 31 CELLS')
print(f'  phi_fit, directional [0,360): R = {Rd:.3f}, p = {pd_:.3f}')
print(f'  fit axis, axial [0,180):      R = {Ra:.3f}, p = {pa:.3f}')
print(f'  (stored 15%-mask phi, directional: R = {Rm:.3f}, p = {pm:.3f};'
      f' theta_envelope axial: R = {Re:.3f}, p = {pe:.3f})')
print(f'  axes within 30 deg of horizontal: {n_hz}/31 (uniform expectation 10.3)')
print('=' * 74)

## S1 gallery reprint

In [ ]:
# 68% intervals for the gallery.  The 95% intervals are dominated by the ~5%
# of fits that land in an alternative basin, so the gallery prints 68%
# intervals; both are kept in per_cell.csv.
k68 = (boot.groupby('cell_id').kappa_dir
           .agg(ci68_lo=lambda x: np.nanpercentile(x, 16),
                ci68_hi=lambda x: np.nanpercentile(x, 84)))
per = per.drop(columns=[c for c in ['ci68_lo', 'ci68_hi'] if c in per]).merge(
    k68, left_on='cell_id', right_index=True)

b_err = boot.assign(err=axial_diff(boot.u_recovered, boot.u_planted))
e68 = b_err.groupby('cell_id').err.apply(lambda x: float(np.nanpercentile(x, 68)))
ori['phi_ci68_halfwidth'] = ori.cell_id.map(e68)

inside = ((per.kappa_dir >= per.ci68_lo) & (per.kappa_dir <= per.ci68_hi)).sum()
print(f'phi 68% half-width: median {ori.phi_ci68_halfwidth.median():.1f} deg, '
      f'max {ori.phi_ci68_halfwidth.max():.1f} deg ({ori.name[ori.phi_ci68_halfwidth.idxmax()]})')
print(f'kappa_dir inside its own 68% interval: {inside}/31')

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from rf_analysis.sparse_noise import gaussian_deriv_m0, gaussian_deriv_m2
MODEL = {0: gaussian_deriv_m0, 1: gaussian_deriv_m1, 2: gaussian_deriv_m2}
CMAP = fs.RF_CMAP if fs else 'RdBu_r'

def render(p, m, px, data_s):
    if m == 0:
        su, sv, th = p['sigma_x'] / px, p['sigma_y'] / px, np.deg2rad(p['theta'])
    else:
        su, sv = p['sigma_phi'] / px, p['sigma_orth'] / px
        th = np.deg2rad(u_from_params(p['theta'], p['kappa_dir']))
    mod = MODEL[m](XY, 1.0, p['x0'] / px, p['y0'] / px, su, sv, th, 0.0).reshape(H, W)
    r = np.corrcoef(mod.ravel(), data_s.ravel())[0, 1]
    return np.sign(r) * mod / np.abs(mod).max(), r**2

gal, gal_chk = {}, []
for _, row in cells.iterrows():
    c, px = row.cell_id, row.pixel_size_deg
    rs = smooth(rf_raw[c]); rec = dataset[[r['cell_id'] for r in dataset].index(c)]
    maps, r2lab = {}, {}
    for m in (0, 2):
        p, q = fit_rf_by_order(np.clip(rf_raw[c], 0, None), np.clip(-rf_raw[c], 0, None),
                               m=m, pixel_size_deg=px, seed=int(c))
        maps[m], r2_refit = render(p, m, px, rs)
        r2lab[m] = rec.get(f'r2_m{m}', r2_refit)      # LABEL with stored R2, not the refit
        gal_chk.append({'name': row['name'], 'm': m,
                        'r2_refit': r2_refit, 'r2_stored': rec.get(f'r2_m{m}', np.nan)})
    fm = fitted_model(row)
    maps[1] = np.sign(np.corrcoef(fm.ravel(), rs.ravel())[0, 1]) * fm / np.abs(fm).max()
    r2lab[1] = row.r_squared
    gal[c] = (rs, maps, r2lab)
gal_chk = pd.DataFrame(gal_chk); gal_chk['gap'] = (gal_chk.r2_refit - gal_chk.r2_stored).abs()
print('m=0/m=2 refit vs stored R2: max gap', round(float(gal_chk.gap.max()), 3))
big = gal_chk[gal_chk.gap > 0.02]
print(big.round(3).to_string(index=False) if len(big) else '  all within 0.02')

In [ ]:
per_i = per.set_index('cell_id'); ori_i = ori.set_index('cell_id')
A4 = (8.27, 11.69)
order = cells.sort_values('kappa_dir').cell_id.tolist()
GAL_PDF = OUT_DIR / 'S1_File_gallery.pdf'
N_PAGES = 5                                   # 31 cells -> 7, 6, 6, 6, 6
ROW_H, TOP0 = 1.52, 0.50

def cover(pdf):
    f = plt.figure(figsize=A4)
    f.text(0.08, 0.92, 'S1 File. Fitted receptive field models for the 31 verified '
           'first-order simple cells', fontsize=13, weight='bold', wrap=True)
    txt = (
      "Each row shows one cell. From left: (1) the raw ridge-regression map; (2) the same map "
      "smoothed at 0.75 pixel, as fitted, with the fitted first-order model's iso-response contours "
      "(solid: ON lobe; dashed: OFF lobe) and an arrow along the differentiation direction, pointing "
      "from the OFF to the ON lobe; (3-5) the idealised Gaussian derivative models of order 0, 1 and 2 "
      "fitted to the same map, with their R\u00b2.\n\n"
      "The first-order model is the stored canonical fit; the zero- and second-order models are "
      "refitted with the same procedure and labelled with their recorded goodness of fit.\n\n"
      "Angles are in degrees, counter-clockwise from rightward horizontal as displayed. "
      "\u03c6 is the differentiation direction of the fitted model on [0\u00b0, 360\u00b0). "
      "\u03c3 is the geometric-mean scale, and \u03ba_dir = \u03c3\u22a5/\u03c3\u03c6 "
      "(values below 1: elongated along the differentiation axis).\n\n"
      "Intervals come from a parametric bootstrap: the cell's fitted model is re-embedded in noise "
      "taken from signal-free lags of its own reconstruction and refitted with the same procedure "
      "(100 draws). The bracket after \u03ba_dir is the central 68% interval (16th to 84th "
      "percentile), and the value after \u03c6 is the 68th percentile of the orientation error. "
      "68% intervals are shown because about 5% of refits converge to an alternative solution, "
      "which dominates the tails; 95% intervals are given in the accompanying table.\n\n"
      "Cells are ordered by \u03ba_dir, as in Fig 4.")
    f.text(0.08, 0.86, txt, fontsize=9.5, va='top', wrap=True)
    pdf.savefig(f); plt.close(f)

def draw_cell(fig, top, c):
    # One cell as a row of five panels; top edge at `top` inches from the page top.
    PW, PH, GAP, LM = 1.44, 1.44 * H / W, 0.07, 0.42
    row = cells.set_index('cell_id').loc[c]; o = ori_i.loc[c]; pr = per_i.loc[c]
    rs, maps, r2lab = gal[c]; raw = rf_raw[c]
    FW, FH = fig.get_size_inches()
    phi_print = int(round(o.phi_fit_vis)) % 360          # 359.8 -> 0, not 360
    head = (f"{row['name']}    cell {c}    container {int(row.container_id)}        "
            f"\u03c3 = {row.sigma_deg:.2f}\u00b0     "
            f"\u03ba_dir = {row.kappa_dir:.2f} [{pr.ci68_lo:.2f}, {pr.ci68_hi:.2f}]     "
            f"\u03c6 = {phi_print}\u00b0 \u00b1 {o.phi_ci68_halfwidth:.0f}\u00b0")
    fig.text(LM / FW, 1 - top / FH, head, fontsize=8.5, weight='bold', va='top', color=C_INK)
    y = top + 0.40 + PH
    axs = [fig.add_axes([(LM + k * (PW + GAP)) / FW, 1 - y / FH, PW / FW, PH / FH])
           for k in range(5)]
    v, vs = np.abs(raw).max(), np.abs(rs).max()
    axs[0].imshow(raw, cmap=CMAP, vmin=-v, vmax=v)
    axs[1].imshow(rs, cmap=CMAP, vmin=-vs, vmax=vs)
    fm = maps[1]
    axs[1].contour(fm, levels=[0.3, 0.6, 0.9], colors=[C_OBS], linewidths=0.8)
    axs[1].contour(fm, levels=[-0.9, -0.6, -0.3], colors=['#1D5C6E'], linewidths=0.8,
                   linestyles='dashed')
    a = np.deg2rad(o.phi_fit_img)
    L = 2.2 * np.sqrt(row.sigma_phi_deg * row.sigma_orth_deg) / row.pixel_size_deg
    axs[1].annotate('', xy=(row.x0_px + L*np.cos(a), row.y0_px + L*np.sin(a)),
                    xytext=(row.x0_px - L*np.cos(a), row.y0_px - L*np.sin(a)),
                    arrowprops=dict(arrowstyle='-|>', color=C_INK, lw=1.2, mutation_scale=9),
                    zorder=10)
    titles = ['raw map', 'smoothed, fitted m=1',
              f'm=0   R\u00b2 = {r2lab[0]:.2f}', f'm=1   R\u00b2 = {r2lab[1]:.2f}',
              f'm=2   R\u00b2 = {r2lab[2]:.2f}']
    for k, m in enumerate((0, 1, 2)):
        axs[2 + k].imshow(maps[m], cmap=CMAP, vmin=-1, vmax=1)
    for ax_, t in zip(axs, titles):
        ax_.set_title(t, fontsize=7.5, loc='center', pad=3)
        ax_.set_xticks([]); ax_.set_yticks([])
        for s_ in ax_.spines.values():
            s_.set_edgecolor('#D8D5D4'); s_.set_linewidth(0.5)
    for ax_ in axs[1:]:
        ax_.set_xlim(axs[0].get_xlim()); ax_.set_ylim(axs[0].get_ylim())

pages = [list(p) for p in np.array_split(order, N_PAGES)]
assert sum(len(p) for p in pages) == 31
with PdfPages(GAL_PDF) as pdf:
    cover(pdf)
    for k, page in enumerate(pages):
        f = plt.figure(figsize=A4)
        for j, c in enumerate(page):
            draw_cell(f, TOP0 + j * ROW_H, int(c))
        f.text(0.5, 0.02, f'S1 File \u2014 page {k + 1}', ha='center',
               fontsize=7, color='#918E8E')
        pdf.savefig(f); plt.close(f)
print(f'wrote {GAL_PDF.name}: cover + {len(pages)} pages '
      f'({", ".join(str(len(p)) for p in pages)} cells)')

In [ ]:
per.merge(ori.drop(columns=['name', 'kappa_dir']), on='cell_id').round(4).to_csv(
    OUT_DIR / 'per_cell.csv', index=False)
gal_chk.round(4).to_csv(OUT_DIR / 'gallery_r2_check.csv', index=False)
lag_tab.round(4).to_csv(OUT_DIR / 'lag_leakage.csv', index=False)
cells.round(5).to_csv(OUT_DIR / 'cell_inputs.csv', index=False)
print('wrote per_cell.csv, lag_leakage.csv, cell_inputs.csv')

# S3 Table: reader-facing per-cell table, ready to drop into the SI as-is
_s3 = (per.merge(ori[['cell_id', 'phi_fit_vis', 'phi_ci68_halfwidth']], on='cell_id')
          .merge(cells[['cell_id', 'container_id', 'sigma_deg']], on='cell_id'))
_s3['phi_deg'] = _s3.phi_fit_vis.round().mod(360).astype(int)
S3 = (_s3.rename(columns={
          'name': 'cell', 'container_id': 'container', 'sigma_deg': 'sigma_deg',
          'ci68_lo': 'kappa_dir_68_lo', 'ci68_hi': 'kappa_dir_68_hi',
          'ci_lo': 'kappa_dir_95_lo', 'ci_hi': 'kappa_dir_95_hi',
          'phi_ci68_halfwidth': 'phi_68_halfwidth_deg',
          'r_squared': 'r2', 'p_vs_round': 'p_vs_round', 'p_bh': 'p_vs_round_BH'})
        [['cell', 'cell_id', 'container', 'sigma_deg', 'kappa_dir',
          'kappa_dir_68_lo', 'kappa_dir_68_hi', 'kappa_dir_95_lo', 'kappa_dir_95_hi',
          'phi_deg', 'phi_68_halfwidth_deg', 'r2', 'p_vs_round', 'p_vs_round_BH']]
        .sort_values('kappa_dir').reset_index(drop=True))
_r3 = ['sigma_deg', 'kappa_dir', 'kappa_dir_68_lo', 'kappa_dir_68_hi',
       'kappa_dir_95_lo', 'kappa_dir_95_hi', 'r2', 'p_vs_round', 'p_vs_round_BH']
S3[_r3] = S3[_r3].round(3)
S3['phi_68_halfwidth_deg'] = S3['phi_68_halfwidth_deg'].round(1)
S3.to_csv(OUT_DIR / 'S3_Table.csv', index=False)
print(f'wrote S3_Table.csv ({len(S3)} rows, {len(S3.columns)} cols): '
      + ', '.join(S3.columns))

In [ ]:
dev = max(np.abs(strf[c][best_lag[c]] - rf_raw[c]).max() / np.abs(rf_raw[c]).max()
          for c in cells.cell_id)
print(f'max deviation, relative to map peak: {dev:.1e}')

In [ ]:
# Binocular vs monocular effect sizes (rank-biserial correlation on kappa_dir,
# the measure used throughout the paper, on the 31 verified cells).
# Zone: binocular = RF-centre azimuth < 40 deg (Methods). r > 0 means the
# binocular zone has the larger values.
from scipy.stats import mannwhitneyu

bz = pd.DataFrame([{'name': r['name'],
                    'azimuth_deg': r['x0_px'] * r['pixel_size_deg'],
                    'sigma_deg':   r['sigma_deg'],
                    'kappa_dir':   r['kappa_dir']} for r in dataset])
bz['zone'] = np.where(bz.azimuth_deg < 40, 'binocular', 'monocular')
B, M = bz[bz.zone == 'binocular'], bz[bz.zone == 'monocular']
print(f"n: binocular {len(B)}, monocular {len(M)}  (azimuth < 40 deg = binocular)")

def rank_biserial(a, b, n_boot=5000, seed=1):
    a, b = np.asarray(a, float), np.asarray(b, float)
    U, p = mannwhitneyu(a, b, alternative='two-sided')
    r = 1 - 2 * U / (len(a) * len(b))            # +: binocular > monocular
    rng = np.random.default_rng(seed)
    bs = [1 - 2 * mannwhitneyu(rng.choice(a, len(a)),
                               rng.choice(b, len(b)))[0] / (len(a) * len(b))
          for _ in range(n_boot)]
    return r, p, np.nanpercentile(bs, [2.5, 97.5])

print('\nbinocular vs monocular (rank-biserial, 31 cells, kappa_dir)')
rows6 = []
for col, label in [('sigma_deg', 'sigma'), ('kappa_dir', 'kappa_dir')]:
    r, p, ci = rank_biserial(B[col], M[col])
    rows6.append({'measure': label, 'p_mannwhitney': p,
                  'rank_biserial': r, 'ci_lo': ci[0], 'ci_hi': ci[1]})
    print(f'  {label:9s}  p = {p:.3f}   r = {r:+.2f}   95% CI [{ci[0]:+.2f}, {ci[1]:+.2f}]')
pd.DataFrame(rows6).round(4).to_csv(OUT_DIR / 'binocular_effect_sizes.csv', index=False)
print(f'\nwrote {OUT_DIR / "binocular_effect_sizes.csv"}')